In [1]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from mrmr import mrmr_classif
import joblib
import numpy as np
import pandas as pd
import random
import time

start_time = time.time()

# ── Load SWELL ────────────────────────────────────────────────────────────────
swell = pd.read_csv("combined-swell-classification-hrv-dataset.csv")

label1 = pd.DataFrame(swell['Condition Label'].map({0:0, 1:1, 2:1}))
subject_id = swell['subject_id']
swell = swell.drop(columns=['subject_id','sampen','higuci','condition','NasaTLX class','Condition Label','NasaTLX Label'])
swell['subject_id'] = subject_id
swell_x = pd.concat([swell, label1], axis=1)

# ── Load WESAD ────────────────────────────────────────────────────────────────
wesad = pd.read_csv("wesad-chest-combined-classification-hrv.csv")

subject_id = pd.DataFrame(wesad['subject id'].map({2:26,3:27,4:28,5:29,6:30,7:31,8:32,9:33,10:34,11:35,13:36,14:37,15:38,16:39,17:40}))
wesad = wesad.rename(columns={"condition label": "Condition Label"})
X_wesad = wesad.iloc[:, 0:-5]
y_wesad = wesad['Condition Label']
X_wesad['subject_id'] = subject_id
label2 = pd.DataFrame(y_wesad.map({1:0, 0:0, 2:1}))
X_wesad = pd.concat([X_wesad, label2], axis=1)

# ── Combine and filter to our 8 features ─────────────────────────────────────
our_features = ['MEAN_RR','MEDIAN_RR','MEAN_RR_SQRT','MEAN_RR_LOG','HR','HR_SQRT','RMSSD','HF']

dfs = [X_wesad[our_features + ['subject_id','Condition Label']],
       swell_x[our_features + ['subject_id','Condition Label']]]
random.shuffle(dfs)
dfs = pd.concat(dfs, axis=0)

print(f"Combined dataset: {len(dfs)} rows, {dfs['subject_id'].nunique()} subjects")

# ── Train ─────────────────────────────────────────────────────────────────────
df = dfs
group_col = 'subject_id'
test_size = 0.3
X_train, X_test, y_train, y_test = [], [], [], []

for group_id, group_data in df.groupby(group_col):
    X_group_train, X_group_test, y_group_train, y_group_test = train_test_split(
        group_data.iloc[:, :-2],
        group_data.iloc[:, -1],
        test_size=test_size,
        random_state=123, stratify=group_data.iloc[:, -1]
    )
    X_train.append(X_group_train)
    X_test.append(X_group_test)
    y_train.append(y_group_train)
    y_test.append(y_group_test)

X_train = pd.concat(X_train)
y_train = pd.concat(y_train)

selected_features = mrmr_classif(X=X_train, y=y_train, K=8)
print("Selected features:", selected_features)
x_train = X_train[selected_features]

kfold = StratifiedKFold(n_splits=15, shuffle=True, random_state=123)
best_model = None
best_score = 0
model = RandomForestClassifier(max_depth=20, min_samples_split=2, n_estimators=100)

for train_index, val_index in kfold.split(x_train, y_train):
    X_train_kf, X_val = np.array(x_train)[train_index], np.array(x_train)[val_index]
    y_train_kf, y_val = np.array(y_train)[train_index], np.array(y_train)[val_index]
    model.fit(X_train_kf, y_train_kf)
    y_pred_val = model.predict(X_val)
    val_score = accuracy_score(y_val, y_pred_val)
    print(f"Validation score: {val_score:.4f}")
    if val_score > best_score:
        best_score = val_score
        best_model = model

for i in range(len(X_test)):
    y_pred = best_model.predict(X_test[i][selected_features])
    test_score = accuracy_score(y_test[i], y_pred)
    print(f"Test score for subject {i+1}: {test_score:.4f}")
    report = classification_report(y_test[i], y_pred)
    print(f"Classification report for subject {i+1}:\n{report}\n")

# ── Save model ────────────────────────────────────────────────────────────────
joblib.dump(best_model, 'stress_model_combined.pkl')
joblib.dump(selected_features, 'selected_features_combined.pkl')
print("Model saved to stress_model_combined.pkl")

end_time = time.time()
print("Computational time:", end_time - start_time)

Combined dataset: 527288 rows, 37 subjects


100%|██████████| 8/8 [00:02<00:00,  2.69it/s]


Selected features: ['RMSSD', 'MEDIAN_RR', 'MEAN_RR', 'MEAN_RR_SQRT', 'MEAN_RR_LOG', 'HR', 'HF', 'HR_SQRT']
Validation score: 0.9919
Validation score: 0.9921
Validation score: 0.9933
Validation score: 0.9924
Validation score: 0.9932
Validation score: 0.9934
Validation score: 0.9931
Validation score: 0.9927
Validation score: 0.9931
Validation score: 0.9936
Validation score: 0.9925
Validation score: 0.9937
Validation score: 0.9920
Validation score: 0.9926
Validation score: 0.9927
Test score for subject 1: 1.0000
Classification report for subject 1:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2952
           1       1.00      1.00      1.00      2802

    accuracy                           1.00      5754
   macro avg       1.00      1.00      1.00      5754
weighted avg       1.00      1.00      1.00      5754


Test score for subject 2: 0.9924
Classification report for subject 2:
              precision    recall  f1-score   supp

In [6]:
import joblib
import numpy as np

# Load model
model = joblib.load('stress_model_combined.pkl')
selected_features = joblib.load('selected_features_combined.pkl')
print("Using features:", selected_features)

# ── Test values — change these ────────────────────────────────────────────────
MEAN_RR      = 829.75
MEDIAN_RR    = 833.33
MEAN_RR_SQRT = 28.80
MEAN_RR_LOG  = 6.72
HR           = 72.36
HR_SQRT      = 8.50
RMSSD        = 42.25
HF           = 0.154
# ─────────────────────────────────────────────────────────────────────────────

# Build input in the same order as selected_features
values = {
    'MEAN_RR':      MEAN_RR,
    'MEDIAN_RR':    MEDIAN_RR,
    'MEAN_RR_SQRT': MEAN_RR_SQRT,
    'MEAN_RR_LOG':  MEAN_RR_LOG,
    'HR':           HR,
    'HR_SQRT':      HR_SQRT,
    'RMSSD':        RMSSD,
    'HF':           HF,
}

features = [[values[f] for f in selected_features]]
confidence = model.predict_proba(features)[0][1]

print(f"\nStress confidence: {confidence:.2%}")
print(f"Prediction:        {'STRESSED' if confidence >= 0.5 else 'NOT STRESSED'}")

Using features: ['RMSSD', 'MEDIAN_RR', 'MEAN_RR', 'MEAN_RR_SQRT', 'MEAN_RR_LOG', 'HR', 'HF', 'HR_SQRT']

Stress confidence: 23.17%
Prediction:        NOT STRESSED


In [ ]:
import joblib
import numpy as np
import neurokit2 as nk

# Load model
model = joblib.load('stress_model_combined.pkl')
selected_features = joblib.load('selected_features_combined.pkl')
print("Using features:", selected_features)

# ── HR readings (BPM) from Apple Watch — change these ────────────────────────
hr_readings = [
    72, 75, 71, 74, 70, 73, 76, 72, 69, 74,
    71, 73, 90, 92, 95, 70, 104, 105, 71, 73,
    70, 72, 75, 71, 74, 69, 73, 76, 70, 72
]
# ─────────────────────────────────────────────────────────────────────────────

# Compute HRV features
rr_intervals = np.array([60000.0 / hr for hr in hr_readings])
peaks = np.cumsum(rr_intervals)

time_feats = nk.hrv_time(peaks=peaks, sampling_rate=1000, show=False)
freq_feats = nk.hrv_frequency(peaks=peaks, sampling_rate=1000, show=False)

mean_rr = float(time_feats["HRV_MeanNN"].iloc[0])
hr_val  = float(np.mean(hr_readings))

values = {
    'MEAN_RR':      round(mean_rr, 4),
    'MEDIAN_RR':    round(float(time_feats["HRV_MedianNN"].iloc[0]), 4),
    'MEAN_RR_SQRT': round(np.sqrt(mean_rr), 4),
    'MEAN_RR_LOG':  round(np.log(mean_rr), 4),
    'HR':           round(hr_val, 4),
    'HR_SQRT':      round(np.sqrt(hr_val), 4),
    'RMSSD':        round(float(time_feats["HRV_RMSSD"].iloc[0]), 4),
    'HF':           round(float(freq_feats["HRV_HF"].iloc[0]), 6),
}

print("\nComputed features:")
for k, v in values.items():
    print(f"  {k:<15} = {v}")

features = [[values[f] for f in selected_features]]
confidence = model.predict_proba(features)[0][1]

print(f"\nStress confidence: {confidence:.2%}")
print(f"Prediction:        {'STRESSED' if confidence >= 0.5 else 'NOT STRESSED'}")

Using features: ['RMSSD', 'MEDIAN_RR', 'MEAN_RR', 'MEAN_RR_SQRT', 'MEAN_RR_LOG', 'HR', 'HF', 'HR_SQRT']

Computed features:
  MEAN_RR         = 792.3495
  MEDIAN_RR       = 821.9178
  MEAN_RR_SQRT    = 28.1487
  MEAN_RR_LOG     = 6.675
  HR              = 76.6
  HR_SQRT         = 8.7521
  RMSSD           = 85.4359
  HF              = 0.02634

Stress confidence: 44.08%
Prediction:        NOT STRESSED
